# Aphasia Treatment Prediction using Multiple Datasets

This notebook demonstrates how to predict aphasia treatment outcomes using functional connectivity data from three different sources:
1. **ARCData** - Resting state data with no longitudinal component
2. **FigShare** - Task-based data with longitudinal component
3. **OSFData** - Resting state data with longitudinal component

We'll use a VAE model to extract features from the functional connectivity data and then predict treatment outcomes.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import our preprocessing module
import preprocessing

## Data Loading and Preprocessing

First, we'll load data from all three sources using our preprocessing module.

In [ ]:
# Load the combined dataset
dataset = preprocessing.prepare_combined_dataset()

# Display information about the datasets
print("Data sources loaded:")
print(f"ARC data: {len(dataset['arc_data']['metadata']) if dataset['arc_data']['metadata'] is not None else 0} subjects")
print(f"FigShare data: {len(dataset['figshare_data']['metadata']) if dataset['figshare_data']['metadata'] is not None else 0} subjects")
print(f"OSF data: {len(dataset['osf_data']['metadata']) if dataset['osf_data']['metadata'] is not None else 0} subjects")

print("\nCombined dataset:")
print(f"Metadata: {len(dataset['metadata'])} subjects")
print(f"FC data: {len(dataset['fc_data'])} subjects")

## Explore Treatment Outcome Data

Let's examine the aphasia treatment outcome data. The Western Aphasia Battery (WAB) Aphasia Quotient (AQ) is commonly used to measure aphasia severity.

In [ ]:
# Examine WAB AQ scores from all data sources
metadata = dataset['metadata']

# Filter for subjects with WAB AQ scores
wab_data = metadata[metadata['wab_aq'].notna()].copy()

# Plot distribution of WAB AQ scores by data source
plt.figure(figsize=(12, 6))
sns.histplot(data=wab_data, x='wab_aq', hue='source', element='step', common_norm=False)
plt.title('Distribution of WAB Aphasia Quotient (AQ) Scores by Data Source')
plt.xlabel('WAB AQ Score')
plt.ylabel('Count')
plt.show()

# Display basic statistics
wab_stats = wab_data.groupby('source')['wab_aq'].describe()
print("WAB AQ Statistics by Source:")
print(wab_stats)

## Prepare Data for VAE Model

Now we'll prepare the data for our VAE model. We'll need to match the functional connectivity data with the metadata.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

# Extract functional connectivity data and match with metadata
fc_data = dataset['fc_data']
fc_subjects = fc_data.index.tolist()

# Filter metadata to include only subjects with FC data
metadata_with_fc = metadata[metadata['subject_id'].isin(fc_subjects)].copy()

# Align FC data with metadata
fc_features = []
targets = []
subject_ids = []
data_sources = []

for _, row in metadata_with_fc.iterrows():
    subject_id = row['subject_id']
    if subject_id in fc_data.index and not pd.isna(row.get('wab_aq', np.nan)):
        # Get FC features (excluding the 'source' column if it exists)
        features = fc_data.loc[subject_id].drop('source', errors='ignore').values
        fc_features.append(features)
        targets.append(row['wab_aq'])
        subject_ids.append(subject_id)
        data_sources.append(row['source'])

fc_features = np.array(fc_features)
targets = np.array(targets)
subject_ids = np.array(subject_ids)
data_sources = np.array(data_sources)

print(f"Prepared dataset: {len(fc_features)} subjects with both FC data and WAB AQ scores")
print(f"Feature dimension: {fc_features.shape[1]}")

## Define VAE Model for Functional Connectivity Data

We'll define a VAE model to learn a latent representation of the functional connectivity data.

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=1000, latent_dim=30, include_covariates=True):
        super(VAE, self).__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.include_covariates = include_covariates
        
        # Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder
        covariate_dims = 3 if include_covariates else 0  # Age, sex, data source
        self.fc3 = nn.Linear(latent_dim + covariate_dims, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)
    
    def encode(self, x):
        h1 = F.relu(self.fc1(x))
        mu = self.fc_mu(h1)
        logvar = self.fc_logvar(h1)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z, covariates=None):
        if self.include_covariates and covariates is not None:
            z = torch.cat([z, covariates], dim=1)
        h3 = F.relu(self.fc3(z))
        return self.fc4(h3)
    
    def forward(self, x, covariates=None):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, covariates), mu, logvar

def loss_function(recon_x, x, mu, logvar, beta=1.0):
    # Reconstruction loss
    BCE = F.mse_loss(recon_x, x, reduction='sum')
    
    # KL divergence loss
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return BCE + beta * KLD

## Train VAE Model

Now we'll train the VAE model on our functional connectivity data.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# Prepare train/test split
X_train, X_test, y_train, y_test, sources_train, sources_test = train_test_split(
    fc_features, targets, data_sources, test_size=0.2, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# One-hot encode data sources
source_encoder = OneHotEncoder(sparse=False)
sources_train_encoded = source_encoder.fit_transform(sources_train.reshape(-1, 1))
sources_test_encoded = source_encoder.transform(sources_test.reshape(-1, 1))

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)
sources_train_tensor = torch.FloatTensor(sources_train_encoded)
sources_test_tensor = torch.FloatTensor(sources_test_encoded)

# Create datasets and dataloaders
train_dataset = TensorDataset(X_train_tensor, sources_train_tensor)
test_dataset = TensorDataset(X_test_tensor, sources_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Initialize model
input_dim = X_train.shape[1]
model = VAE(input_dim=input_dim, hidden_dim=1000, latent_dim=30, include_covariates=True)

# Training parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 10  # Reduced for demonstration purposes

# Training loop
model.train()
train_losses = []

for epoch in range(epochs):
    total_loss = 0
    for batch_idx, (data, covariates) in enumerate(train_loader):
        data = data.to(device)
        covariates = covariates.to(device)
        
        optimizer.zero_grad()
        recon_batch, mu, logvar = model(data, covariates)
        loss = loss_function(recon_batch, data, mu, logvar)
        loss.backward()
        total_loss += loss.item()
        optimizer.step()
    
    avg_loss = total_loss / len(train_loader.dataset)
    train_losses.append(avg_loss)
    
    print(f'Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}')

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses)
plt.title('VAE Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

## Extract Latent Features for Treatment Prediction

Now we'll extract the latent features from the VAE and use them to predict treatment outcomes.

In [ ]:
# Extract latent features
model.eval()
with torch.no_grad():
    # Extract for training data
    train_mu, train_logvar = model.encode(X_train_tensor.to(device))
    train_z = model.reparameterize(train_mu, train_logvar)
    train_latent = train_z.cpu().numpy()
    
    # Extract for testing data
    test_mu, test_logvar = model.encode(X_test_tensor.to(device))
    test_z = model.reparameterize(test_mu, test_logvar)
    test_latent = test_z.cpu().numpy()

print(f"Extracted latent features shape: {train_latent.shape}")

## Visualize Latent Space

Let's visualize the latent space to see how the data is organized.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Apply PCA for visualization
pca = PCA(n_components=2)
train_pca = pca.fit_transform(train_latent)
test_pca = pca.transform(test_latent)

# Combine for plotting
all_pca = np.vstack([train_pca, test_pca])
all_sources = np.concatenate([sources_train, sources_test])
all_targets = np.concatenate([y_train, y_test])

# Create a colormap for WAB AQ scores
plt.figure(figsize=(12, 10))

# Plot by data source
plt.subplot(2, 1, 1)
for source in np.unique(all_sources):
    mask = all_sources == source
    plt.scatter(all_pca[mask, 0], all_pca[mask, 1], alpha=0.7, label=source)
plt.title('PCA of Latent Space by Data Source')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()

# Plot by WAB AQ score
plt.subplot(2, 1, 2)
scatter = plt.scatter(all_pca[:, 0], all_pca[:, 1], c=all_targets, alpha=0.7, cmap='viridis')
plt.colorbar(scatter, label='WAB AQ Score')
plt.title('PCA of Latent Space by WAB AQ Score')
plt.xlabel('PC1')
plt.ylabel('PC2')

plt.tight_layout()
plt.show()

## Build and Evaluate Treatment Prediction Models

Now let's build models to predict aphasia treatment outcomes using the latent features.

In [ ]:
from sklearn.linear_model import Ridge, LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import math

# Combine latent features with covariates
train_covariates_np = sources_train_encoded
test_covariates_np = sources_test_encoded

X_train_combined = np.hstack([train_latent, train_covariates_np])
X_test_combined = np.hstack([test_latent, test_covariates_np])

# Define models to evaluate
models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Linear Regression': LinearRegression(),
    'Elastic Net': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate each model
results = {}

for name, model in models.items():
    # Train the model
    model.fit(X_train_combined, y_train)
    
    # Make predictions
    train_pred = model.predict(X_train_combined)
    test_pred = model.predict(X_test_combined)
    
    # Calculate metrics
    train_rmse = math.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = math.sqrt(mean_squared_error(y_test, test_pred))
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    
    results[name] = {
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'test_pred': test_pred
    }

# Display results
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Train RMSE': [results[m]['train_rmse'] for m in results],
    'Test RMSE': [results[m]['test_rmse'] for m in results],
    'Train R²': [results[m]['train_r2'] for m in results],
    'Test R²': [results[m]['test_r2'] for m in results],
    'Train MAE': [results[m]['train_mae'] for m in results],
    'Test MAE': [results[m]['test_mae'] for m in results],
})

print("Model Evaluation Results:")
print(results_df)

## Visualize Prediction Results

Let's visualize the prediction results for the best performing model.

In [ ]:
# Find the best model based on Test RMSE
best_model_name = results_df.loc[results_df['Test RMSE'].idxmin(), 'Model']
best_model_results = results[best_model_name]
best_model = models[best_model_name]

print(f"Best model: {best_model_name}")
print(f"Test RMSE: {best_model_results['test_rmse']:.2f}")
print(f"Test R²: {best_model_results['test_r2']:.2f}")
print(f"Test MAE: {best_model_results['test_mae']:.2f}")

# Plot actual vs predicted values
plt.figure(figsize=(10, 6))
plt.scatter(y_test, best_model_results['test_pred'], alpha=0.7)
plt.plot([0, 100], [0, 100], 'r--')  # Perfect prediction line
plt.title(f'Actual vs Predicted WAB AQ Scores ({best_model_name})')
plt.xlabel('Actual WAB AQ')
plt.ylabel('Predicted WAB AQ')
plt.grid(True, alpha=0.3)
plt.show()

## Analyze Feature Importance

Let's analyze the feature importance if our best model supports it.

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    # Get feature importances
    importances = best_model.feature_importances_
    
    # Create feature names
    feature_names = [f'Latent_{i}' for i in range(train_latent.shape[1])]
    feature_names.extend([f'Source_{i}' for i in range(train_covariates_np.shape[1])])
    
    # Sort by importance
    indices = np.argsort(importances)[::-1]
    
    # Plot top 20 features
    plt.figure(figsize=(12, 8))
    plt.title(f'Feature Importances ({best_model_name})')
    plt.bar(range(20), importances[indices][:20], align='center')
    plt.xticks(range(20), [feature_names[i] for i in indices][:20], rotation=90)
    plt.tight_layout()
    plt.show()
    
    # Display top 10 features
    top_features = pd.DataFrame({
        'Feature': [feature_names[i] for i in indices][:10],
        'Importance': importances[indices][:10]
    })
    print("Top 10 Important Features:")
    print(top_features)
elif isinstance(best_model, LinearRegression) or isinstance(best_model, Ridge) or isinstance(best_model, ElasticNet):
    # For linear models, we can look at coefficients
    coef = best_model.coef_
    
    # Create feature names
    feature_names = [f'Latent_{i}' for i in range(train_latent.shape[1])]
    feature_names.extend([f'Source_{i}' for i in range(train_covariates_np.shape[1])])
    
    # Sort by absolute coefficient value
    indices = np.argsort(np.abs(coef))[::-1]
    
    # Plot top 20 features
    plt.figure(figsize=(12, 8))
    plt.title(f'Feature Coefficients ({best_model_name})')
    plt.bar(range(20), coef[indices][:20], align='center')
    plt.xticks(range(20), [feature_names[i] for i in indices][:20], rotation=90)
    plt.tight_layout()
    plt.show()
    
    # Display top 10 features
    top_features = pd.DataFrame({
        'Feature': [feature_names[i] for i in indices][:10],
        'Coefficient': coef[indices][:10]
    })
    print("Top 10 Features by Coefficient Magnitude:")
    print(top_features)
else:
    print(f"Feature importance analysis not available for {best_model_name}")

## Predict Treatment Recovery Trajectories

Using the longitudinal data from FigShare and OSF, let's predict recovery trajectories.

In [ ]:
# Look for subjects with longitudinal data
longitudinal_subjects = metadata[metadata['is_longitudinal'] == True]['subject_id'].unique()
print(f"Found {len(longitudinal_subjects)} subjects with longitudinal data")

# For demonstration purposes, let's simulate recovery trajectories
# In a real scenario, you would use actual longitudinal data points

# Select subjects with different baseline WAB AQ scores
wab_range = metadata['wab_aq'].max() - metadata['wab_aq'].min()
low_threshold = metadata['wab_aq'].min() + wab_range / 3
high_threshold = metadata['wab_aq'].max() - wab_range / 3

low_subjects = metadata[(metadata['wab_aq'] < low_threshold) & 
                         (metadata['is_longitudinal'] == True)]['subject_id'].iloc[:3].tolist()
mid_subjects = metadata[(metadata['wab_aq'] >= low_threshold) & 
                         (metadata['wab_aq'] < high_threshold) & 
                         (metadata['is_longitudinal'] == True)]['subject_id'].iloc[:3].tolist()
high_subjects = metadata[(metadata['wab_aq'] >= high_threshold) & 
                          (metadata['is_longitudinal'] == True)]['subject_id'].iloc[:3].tolist()

sample_subjects = low_subjects + mid_subjects + high_subjects

# Get baseline scores
baseline_scores = []
for subject in sample_subjects:
    score = metadata[metadata['subject_id'] == subject]['wab_aq'].iloc[0]
    baseline_scores.append(score)

# Simulate trajectories based on baseline scores
time_points = np.array([0, 30, 90, 180, 365])  # Days post stroke
trajectories = []

for i, baseline in enumerate(baseline_scores):
    max_improvement = min(100 - baseline, 40)  # Maximum improvement is 40 points or up to 100
    k = 0.01 if baseline < 50 else 0.02  # Rate parameter - higher baseline means faster recovery
    noise = np.random.normal(0, 2, len(time_points))  # Add some noise
    
    # Recovery curve: baseline + max_improvement * (1 - exp(-k * t))
    trajectory = baseline + max_improvement * (1 - np.exp(-k * time_points)) + noise
    trajectory = np.clip(trajectory, 0, 100)  # Ensure scores are between 0 and 100
    trajectories.append(trajectory)

# Plot recovery trajectories
plt.figure(figsize=(12, 8))
for i, (subject, trajectory) in enumerate(zip(sample_subjects, trajectories)):
    baseline = baseline_scores[i]
    label = f"{subject} (Baseline: {baseline:.1f})"
    
    # Determine line style based on baseline score group
    if i < len(low_subjects):
        linestyle = '--'
    elif i < len(low_subjects) + len(mid_subjects):
        linestyle = '-'
    else:
        linestyle = '-.'
    
    plt.plot(time_points, trajectory, marker='o', label=label, linestyle=linestyle)

plt.axhline(y=93.8, color='r', linestyle='--', label='Normal Cutoff (93.8)')
plt.title('Simulated Aphasia Recovery Trajectories')
plt.xlabel('Days Post Stroke')
plt.ylabel('WAB AQ Score')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusion

In this notebook, we've demonstrated how to combine data from three different sources (ARCData, FigShare, and OSFData) to predict aphasia treatment outcomes. We've:

1. Loaded and preprocessed data from all three sources
2. Built a VAE model to extract latent features from functional connectivity data
3. Used the latent features to predict WAB AQ scores
4. Simulated recovery trajectories based on baseline scores

This approach shows the potential of using multimodal data integration and deep learning to predict treatment outcomes in aphasia rehabilitation.